In [2]:
#import the pdf:
# for local file
from magic_doc.docconv import DocConverter
converter = DocConverter(s3_config=None)
markdown_content, time_cost = converter.convert("/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/Binder1.pdf", conv_timeout=300)

print(markdown_content)





2024-07-19 12:04:43.492 | INFO     | magic_pdf.libs.pdf_check:detect_invalid_chars:57 - cid_count: 0, text_len: 49679, cid_chars_radio: 0.0
2024-07-19 12:04:43.566 | INFO     | magic_doc.contrib.pdf.pdf_extractor:run:70 - stream io data is digital pdf


Call: [HORIZON-CL6-2024-FARM2FORK-01-1] — [Agro-pastoral/outdoor livestock systems and wildlife management] 

Part B - Page 1 of 50 

COHABITATION AND OPTIMAL RECONCILIATION: INTEGRATING TERRITORIAL TRANSFORMATIONS AND ECOLOGICAL RESILIENCE

CRITTER  

[This document is tagged. Do not delete the tags; they are needed for processing.] #@APP-FORM-HERIAIA@# List of participants

PARTICIPANT NO. PARTICIPANT ORGANISATION NAME SHORT NAME COUNTRY

1 (Coordinator) Syddansk Universitet SDU DK

2 Lapin Yliopisto UOL FI

3 Università degli Studi di Padova UNIPD IT

4 Centro de Investigacion y Tecnologia Agroalimentaria de Aragon CITA ES

5 Association WWF Bulgaria WWF-BG BG

6 WWF Slovensko WWF-SK SK

7 Zavod za Gozdove Slovenije - Slovenia Forest Service SFS SI

8 Schola Campesina APS CAMP IT

9 De Surdurulebilir Enerji ve Insaat Sanayi Ticaret Limited Sirketi DEM TR

10 Luonnonvarakeskus - Natural Resources Institute Finland LUKE FI

11 Interspread GmbH INSP AT

12 Wageningen University & Resea

In [9]:
import ollama
from nltk import word_tokenize
import re
import math

# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'Mathstral',
        'temperature': 0.3,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'Qwen2',
        'temperature': 0.6,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'Qwen2',
        'temperature': 0.8,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'internlm2',
        'temperature': 0.5,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Introduction and Excellence": {
        'Highly analytical evaluator': "Does the document present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent?",
        'Collaboration expert': "Does the methodology involve appropriate interdisciplinary approaches and demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships?",
        'Innovation and impact specialist': "Does the document clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies, and provide realistic and achievable objectives that could generate significant impact?",
        'Project management expert': "Does the work plan provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Impact": {
        'Highly analytical evaluator': "Are the pathways to achieve the expected outcomes and impacts well-defined and credible? Are the measures to maximize these impacts suitable, realistic, and well-described with quantifiable estimates?",
        'Collaboration expert': "Does the document provide detailed plans for stakeholder engagement, dissemination, and exploitation activities that ensure broad uptake and impact of the project results among relevant target groups?",
        'Innovation and impact specialist': "Does the project have the potential to drive significant scientific, economic, and societal impacts? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Are the dissemination, exploitation, and communication plans detailed and well-structured, with clear objectives, target groups, and measures to ensure effective communication and uptake of the project results?"
    },
    "Quality and Efficiency of the Implementation": {
        'Highly analytical evaluator': "Is the work plan detailed and efficient, with appropriate allocation of resources and efforts to each work package? Are risks identified and mitigation measures well-defined and realistic?",
        'Collaboration expert': "Does the consortium have the necessary expertise and infrastructure to achieve the project’s objectives? How well do the members complement each other and cover the value chain, ensuring effective collaboration and resource utilization?",
        'Innovation and impact specialist': "Does the project demonstrate robust innovation management and risk mitigation strategies? Are potential barriers identified, and are the proposed mitigation measures adequate to ensure successful implementation and impact of the project?",
        'Project management expert': "Does the work plan provide a comprehensive overview with clear roles and responsibilities for each participant? Are the timelines, milestones, and deliverables well-defined and aligned with the project objectives to ensure efficient implementation and monitoring?"
    }
}

def split_document(text):
    sections = re.split(r'(?i)2\. impact', text)
    if len(sections) > 1:
        sections[1] = '2. Impact' + sections[1]
    else:
        raise ValueError("Section '2. Impact' not found in the document.")
    
    sections = [re.split(r'(?i)3\. quality and efficiency of the implementation', section) for section in sections]
    flattened_sections = [item for sublist in sections for item in sublist]
    if len(flattened_sections) == 3:
        flattened_sections[2] = '3. Quality And Efficiency Of The Implementation' + flattened_sections[2]
    else:
        raise ValueError("Section '3. Quality And Efficiency Of The Implementation' not found in the document.")
    
    return flattened_sections

def calculate_context_window(sections):
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are {personality_key}, an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    print(f"Sending to model: {section[:500]}...")  # Print the first 500 characters of the section for debugging
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

def test_personality(personality_key):
    params = personalities_parameters[personality_key]
    document_text = markdown_content  # Assign your document content here
    sections = split_document(document_text)
    context_window = calculate_context_window(sections)
    section_titles = [
        "Introduction and Excellence",
        "Impact",
        "Quality and Efficiency of the Implementation"
    ]
    
    answers = []
    for section_title, section in zip(section_titles, sections):
        print(f"Analyzing Section: {section_title}")
        answer = analyze_section(personality_key, params, section_title, section, context_window)
        answers.append(f"Section {section_title}:\n{answer}")
    
    return answers

# Example usage:
# To test a specific personality and print the results
personality_to_test = 'Project management expert'
results = test_personality(personality_to_test)
for i, result in enumerate(results):
    print(f"Section {i+1} - {result}\n")


Analyzing Section: Introduction and Excellence
Sending to model: Call: [HORIZON-CL6-2024-FARM2FORK-01-1] — [Agro-pastoral/outdoor livestock systems and wildlife management] 

Part B - Page 1 of 50 

COHABITATION AND OPTIMAL RECONCILIATION: INTEGRATING TERRITORIAL TRANSFORMATIONS AND ECOLOGICAL RESILIENCE

CRITTER  

[This document is tagged. Do not delete the tags; they are needed for processing.] #@APP-FORM-HERIAIA@# List of participants

PARTICIPANT NO. PARTICIPANT ORGANISATION NAME SHORT NAME COUNTRY

1 (Coordinator) Syddansk Universitet SDU DK

2 Lapin Yl...
Analyzing Section: Impact
Sending to model: 2. Impact #@IMP-ACT-IA@#

CRITTER will provide impactful results in the short (by the end of the project), medium (3-5 years after the end of the project) and long term (>10 years after the end of the project) for rural and wildlife ecosystems. The project is geared towards benefiting a broad diversity of stakeholders at local, regional, national, and European level, tools for the app

In [11]:
import ollama
from nltk import word_tokenize
import re
import math

# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'Mathstral',
        'temperature': 0.3,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'Qwen2',
        'temperature': 0.6,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'Qwen2',
        'temperature': 0.8,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'internlm2',
        'temperature': 0.5,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Introduction and Excellence": {
        'Highly analytical evaluator': "Does the document present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent?",
        'Collaboration expert': "Does the methodology involve appropriate interdisciplinary approaches and demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships?",
        'Innovation and impact specialist': "Does the document clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies, and provide realistic and achievable objectives that could generate significant impact?",
        'Project management expert': "Does the work plan provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Impact": {
        'Highly analytical evaluator': "Are the pathways to achieve the expected outcomes and impacts well-defined and credible? Are the measures to maximize these impacts suitable, realistic, and well-described with quantifiable estimates?",
        'Collaboration expert': "Does the document provide detailed plans for stakeholder engagement, dissemination, and exploitation activities that ensure broad uptake and impact of the project results among relevant target groups?",
        'Innovation and impact specialist': "Does the project have the potential to drive significant scientific, economic, and societal impacts? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Are the dissemination, exploitation, and communication plans detailed and well-structured, with clear objectives, target groups, and measures to ensure effective communication and uptake of the project results?"
    },
    "Quality and Efficiency of the Implementation": {
        'Highly analytical evaluator': "Is the work plan detailed and efficient, with appropriate allocation of resources and efforts to each work package? Are risks identified and mitigation measures well-defined and realistic?",
        'Collaboration expert': "Does the consortium have the necessary expertise and infrastructure to achieve the project’s objectives? How well do the members complement each other and cover the value chain, ensuring effective collaboration and resource utilization?",
        'Innovation and impact specialist': "Does the project demonstrate robust innovation management and risk mitigation strategies? Are potential barriers identified, and are the proposed mitigation measures adequate to ensure successful implementation and impact of the project?",
        'Project management expert': "Does the work plan provide a comprehensive overview with clear roles and responsibilities for each participant? Are the timelines, milestones, and deliverables well-defined and aligned with the project objectives to ensure efficient implementation and monitoring?"
    }
}

def split_document(text):
    sections = re.split(r'(?i)2\. impact', text)
    if len(sections) > 1:
        sections[1] = '2. Impact' + sections[1]
    else:
        raise ValueError("Section '2. Impact' not found in the document.")
    
    sections = [re.split(r'(?i)3\. quality and efficiency of the implementation', section) for section in sections]
    flattened_sections = [item for sublist in sections for item in sublist]
    if len(flattened_sections) == 3:
        flattened_sections[2] = '3. Quality And Efficiency Of The Implementation' + flattened_sections[2]
    else:
        raise ValueError("Section '3. Quality And Efficiency Of The Implementation' not found in the document.")
    
    return flattened_sections

def calculate_context_window(sections):
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are {personality_key}, an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

def evaluate_all_experts(document_text):
    sections = split_document(document_text)
    context_window = calculate_context_window(sections)
    section_titles = [
        "Introduction and Excellence",
        "Impact",
        "Quality and Efficiency of the Implementation"
    ]

    all_expert_responses = []

    for personality_key, params in personalities_parameters.items():
        expert_responses = {"personality": personality_key, "responses": {}}
        for section_title, section in zip(section_titles, sections):
            print(f"Analyzing Section: {section_title} with {personality_key}")
            answer = analyze_section(personality_key, params, section_title, section, context_window)
            expert_responses["responses"][section_title] = answer
        all_expert_responses.append(expert_responses)
    
    return all_expert_responses

def display_responses(all_expert_responses):
    for expert in all_expert_responses:
        personality = expert["personality"]
        print(f"\n\n=== {personality} ===")
        for section_title, response in expert["responses"].items():
            print(f"\nSection {section_title}:\n{response}\n")

# Example usage:
document_text = markdown_content  # Assign your document content here
all_expert_responses = evaluate_all_experts(document_text)
display_responses(all_expert_responses)


Analyzing Section: Introduction and Excellence with Highly analytical evaluator
Analyzing Section: Impact with Highly analytical evaluator
Analyzing Section: Quality and Efficiency of the Implementation with Highly analytical evaluator
Analyzing Section: Introduction and Excellence with Collaboration expert
Analyzing Section: Impact with Collaboration expert
Analyzing Section: Quality and Efficiency of the Implementation with Collaboration expert
Analyzing Section: Introduction and Excellence with Innovation and impact specialist
Analyzing Section: Impact with Innovation and impact specialist
Analyzing Section: Quality and Efficiency of the Implementation with Innovation and impact specialist
Analyzing Section: Introduction and Excellence with Project management expert
Analyzing Section: Impact with Project management expert
Analyzing Section: Quality and Efficiency of the Implementation with Project management expert


=== Highly analytical evaluator ===

Section Introduction and Exce

In [12]:
import ollama
from nltk import word_tokenize
import re
import math

# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'Mathstral',
        'temperature': 0.3,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'Qwen2',
        'temperature': 0.6,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'Qwen2',
        'temperature': 0.8,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'internlm2',
        'temperature': 0.5,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Introduction and Excellence": {
        'Highly analytical evaluator': "Does the document present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent?",
        'Collaboration expert': "Does the methodology involve appropriate interdisciplinary approaches and demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships?",
        'Innovation and impact specialist': "Does the document clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies, and provide realistic and achievable objectives that could generate significant impact?",
        'Project management expert': "Does the work plan provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Impact": {
        'Highly analytical evaluator': "Are the pathways to achieve the expected outcomes and impacts well-defined and credible? Are the measures to maximize these impacts suitable, realistic, and well-described with quantifiable estimates?",
        'Collaboration expert': "Does the document provide detailed plans for stakeholder engagement, dissemination, and exploitation activities that ensure broad uptake and impact of the project results among relevant target groups?",
        'Innovation and impact specialist': "Does the project have the potential to drive significant scientific, economic, and societal impacts? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Are the dissemination, exploitation, and communication plans detailed and well-structured, with clear objectives, target groups, and measures to ensure effective communication and uptake of the project results?"
    },
    "Quality and Efficiency of the Implementation": {
        'Highly analytical evaluator': "Is the work plan detailed and efficient, with appropriate allocation of resources and efforts to each work package? Are risks identified and mitigation measures well-defined and realistic?",
        'Collaboration expert': "Does the consortium have the necessary expertise and infrastructure to achieve the project’s objectives? How well do the members complement each other and cover the value chain, ensuring effective collaboration and resource utilization?",
        'Innovation and impact specialist': "Does the project demonstrate robust innovation management and risk mitigation strategies? Are potential barriers identified, and are the proposed mitigation measures adequate to ensure successful implementation and impact of the project?",
        'Project management expert': "Does the work plan provide a comprehensive overview with clear roles and responsibilities for each participant? Are the timelines, milestones, and deliverables well-defined and aligned with the project objectives to ensure efficient implementation and monitoring?"
    }
}

def split_document(text):
    sections = re.split(r'(?i)2\. impact', text)
    if len(sections) > 1:
        sections[1] = '2. Impact' + sections[1]
    else:
        raise ValueError("Section '2. Impact' not found in the document.")
    
    sections = [re.split(r'(?i)3\. quality and efficiency of the implementation', section) for section in sections]
    flattened_sections = [item for sublist in sections for item in sublist]
    if len(flattened_sections) == 3:
        flattened_sections[2] = '3. Quality And Efficiency Of The Implementation' + flattened_sections[2]
    else:
        raise ValueError("Section '3. Quality And Efficiency Of The Implementation' not found in the document.")
    
    return flattened_sections

def calculate_context_window(sections):
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are {personality_key}, an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

def evaluate_all_experts(document_text):
    sections = split_document(document_text)
    context_window = calculate_context_window(sections)
    section_titles = [
        "Introduction and Excellence",
        "Impact",
        "Quality and Efficiency of the Implementation"
    ]

    all_expert_responses = []

    for personality_key, params in personalities_parameters.items():
        expert_responses = {"personality": personality_key, "responses": {}}
        for section_title, section in zip(section_titles, sections):
            print(f"The {personality_key} is analyzing the {section_title} section")
            answer = analyze_section(personality_key, params, section_title, section, context_window)
            expert_responses["responses"][section_title] = answer
        all_expert_responses.append(expert_responses)
    
    return all_expert_responses

def display_responses(all_expert_responses):
    for expert in all_expert_responses:
        personality = expert["personality"]
        print(f"\n\n=== {personality} ===")
        for section_title, response in expert["responses"].items():
            print(f"\nSection {section_title}:\n{response}\n")

def iterate_experts(all_expert_responses):
    sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
    consolidated_outputs = {section: [] for section in sections}
    
    for expert in all_expert_responses:
        personality = expert["personality"]
        for section in sections:
            if section in expert["responses"]:
                consolidated_outputs[section].append(f"=== {personality} ===\n\n{expert['responses'][section]}\n")
    
    return consolidated_outputs

document_text = markdown_content  # Assign your document content here
all_expert_responses = evaluate_all_experts(document_text)
consolidated_outputs = iterate_experts(all_expert_responses)

# Display consolidated outputs
for section, reviews in consolidated_outputs.items():
    print(f"\n\n=== Consolidated Reviews for {section} ===\n")
    for review in reviews:
        print(review)


Analyzing Section: Introduction and Excellence with Highly analytical evaluator
Analyzing Section: Impact with Highly analytical evaluator
Analyzing Section: Quality and Efficiency of the Implementation with Highly analytical evaluator
Analyzing Section: Introduction and Excellence with Collaboration expert
Analyzing Section: Impact with Collaboration expert
Analyzing Section: Quality and Efficiency of the Implementation with Collaboration expert
Analyzing Section: Introduction and Excellence with Innovation and impact specialist
Analyzing Section: Impact with Innovation and impact specialist
Analyzing Section: Quality and Efficiency of the Implementation with Innovation and impact specialist
Analyzing Section: Introduction and Excellence with Project management expert
Analyzing Section: Impact with Project management expert
Analyzing Section: Quality and Efficiency of the Implementation with Project management expert


=== Consolidated Reviews for Introduction and Excellence ===

=== 

In [13]:
import ollama
from nltk import word_tokenize
import re
import math

# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'Mathstral',
        'temperature': 0.3,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'Qwen2',
        'temperature': 0.6,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'Qwen2',
        'temperature': 0.8,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'internlm2',
        'temperature': 0.5,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Introduction and Excellence": {
        'Highly analytical evaluator': "Does the document present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent?",
        'Collaboration expert': "Does the methodology involve appropriate interdisciplinary approaches and demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships?",
        'Innovation and impact specialist': "Does the document clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies, and provide realistic and achievable objectives that could generate significant impact?",
        'Project management expert': "Does the work plan provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Impact": {
        'Highly analytical evaluator': "Are the pathways to achieve the expected outcomes and impacts well-defined and credible? Are the measures to maximize these impacts suitable, realistic, and well-described with quantifiable estimates?",
        'Collaboration expert': "Does the document provide detailed plans for stakeholder engagement, dissemination, and exploitation activities that ensure broad uptake and impact of the project results among relevant target groups?",
        'Innovation and impact specialist': "Does the project have the potential to drive significant scientific, economic, and societal impacts? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Are the dissemination, exploitation, and communication plans detailed and well-structured, with clear objectives, target groups, and measures to ensure effective communication and uptake of the project results?"
    },
    "Quality and Efficiency of the Implementation": {
        'Highly analytical evaluator': "Is the work plan detailed and efficient, with appropriate allocation of resources and efforts to each work package? Are risks identified and mitigation measures well-defined and realistic?",
        'Collaboration expert': "Does the consortium have the necessary expertise and infrastructure to achieve the project’s objectives? How well do the members complement each other and cover the value chain, ensuring effective collaboration and resource utilization?",
        'Innovation and impact specialist': "Does the project demonstrate robust innovation management and risk mitigation strategies? Are potential barriers identified, and are the proposed mitigation measures adequate to ensure successful implementation and impact of the project?",
        'Project management expert': "Does the work plan provide a comprehensive overview with clear roles and responsibilities for each participant? Are the timelines, milestones, and deliverables well-defined and aligned with the project objectives to ensure efficient implementation and monitoring?"
    }
}

def split_document(text):
    sections = re.split(r'(?i)2\. impact', text)
    if len(sections) > 1:
        sections[1] = '2. Impact' + sections[1]
    else:
        raise ValueError("Section '2. Impact' not found in the document.")
    
    sections = [re.split(r'(?i)3\. quality and efficiency of the implementation', section) for section in sections]
    flattened_sections = [item for sublist in sections for item in sublist]
    if len(flattened_sections) == 3:
        flattened_sections[2] = '3. Quality And Efficiency Of The Implementation' + flattened_sections[2]
    else:
        raise ValueError("Section '3. Quality And Efficiency Of The Implementation' not found in the document.")
    
    return flattened_sections

def calculate_context_window(sections):
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are {personality_key}, an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

def evaluate_all_experts(document_text):
    sections = split_document(document_text)
    context_window = calculate_context_window(sections)
    section_titles = [
        "Introduction and Excellence",
        "Impact",
        "Quality and Efficiency of the Implementation"
    ]

    all_expert_responses = []

    for personality_key, params in personalities_parameters.items():
        expert_responses = {"personality": personality_key, "responses": {}}
        for section_title, section in zip(section_titles, sections):
            print(f"The {personality_key} is analyzing the {section_title} section")
            answer = analyze_section(personality_key, params, section_title, section, context_window)
            expert_responses["responses"][section_title] = answer
        all_expert_responses.append(expert_responses)
    
    return all_expert_responses

def synthesize_feedback(all_expert_responses):
    sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
    consolidated_feedback = {section: {"strengths": [], "weaknesses": [], "recommendations": [], "summaries": []} for section in sections}
    
    for expert in all_expert_responses:
        for section in sections:
            if section in expert["responses"]:
                response = expert["responses"][section]
                # Extracting strengths, weaknesses, recommendations, and summaries from each expert's response
                strengths = re.findall(r'\*\*Strengths\*\*:\n((?:\s+- .*\n)+)', response)
                weaknesses = re.findall(r'\*\*Weaknesses\*\*:\n((?:\s+- .*\n)+)', response)
                recommendations = re.findall(r'\*\*Recommendations\*\*:\n((?:\s+- .*\n)+)', response)
                summaries = re.findall(r'\*\*Summary\*\*:\n((?:\s+- .*\n)+)', response)

                if strengths:
                    consolidated_feedback[section]["strengths"].extend(strengths[0].strip().split('\n'))
                if weaknesses:
                    consolidated_feedback[section]["weaknesses"].extend(weaknesses[0].strip().split('\n'))
                if recommendations:
                    consolidated_feedback[section]["recommendations"].extend(recommendations[0].strip().split('\n'))
                if summaries:
                    consolidated_feedback[section]["summaries"].extend(summaries[0].strip().split('\n'))

    return consolidated_feedback

def generate_review(consolidated_feedback):
    final_review = []

    for section, feedback in consolidated_feedback.items():
        final_review.append(f"\n\n=== Synthesis for {section} ===\n")

        if feedback["strengths"]:
            final_review.append("\n**Strengths**:\n")
            final_review.extend(feedback["strengths"])

        if feedback["weaknesses"]:
            final_review.append("\n**Weaknesses**:\n")
            final_review.extend(feedback["weaknesses"])

        if feedback["recommendations"]:
            final_review.append("\n**Recommendations**:\n")
            final_review.extend(feedback["recommendations"])

        if feedback["summaries"]:
            final_review.append("\n**Summary**:\n")
            final_review.extend(feedback["summaries"])

    return "\n".join(final_review)

def analyze_reviewer(section_title, consolidated_feedback):
    context_window = 5000
    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback: {consolidated_feedback[section_title]}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="llama3-groq-tool-use", messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

document_text = markdown_content  # Assign your document content here
all_expert_responses = evaluate_all_experts(document_text)
consolidated_feedback = synthesize_feedback(all_expert_responses)
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, consolidated_feedback))

print("\n\n".join(final_review))


The Highly analytical evaluator is analyzing the Introduction and Excellence section
The Highly analytical evaluator is analyzing the Impact section
The Highly analytical evaluator is analyzing the Quality and Efficiency of the Implementation section
The Collaboration expert is analyzing the Introduction and Excellence section
The Collaboration expert is analyzing the Impact section
The Collaboration expert is analyzing the Quality and Efficiency of the Implementation section
The Innovation and impact specialist is analyzing the Introduction and Excellence section
The Innovation and impact specialist is analyzing the Impact section
The Innovation and impact specialist is analyzing the Quality and Efficiency of the Implementation section
The Project management expert is analyzing the Introduction and Excellence section
The Project management expert is analyzing the Impact section
The Project management expert is analyzing the Quality and Efficiency of the Implementation section
Section 

In [16]:
# print all_expert_responses in a text format for easy reading
for expert in all_expert_responses:
    print(f"\n\n=== {expert['personality']} ===")
    for section_title, response in expert['responses'].items():
        print(f"\nSection {section_title}:\n{response}\n")



=== Highly analytical evaluator ===

Section Introduction and Excellence:
1. The problem statement is clearly defined: "Identify whether a given number (n) can be expressed as an integer linear combination of two other integers."
2. Clear objectives are stated in the abstract: To determine if n can be written as \( ax + by \), where x and y are non-negative integers, with gcd(a, b)=1.
3. The methodology is sound because it uses a mathematical approach based on number theory principles to solve this problem effectively using the Extended Euclidean Algorithm (EEA). 4. All underlying concepts such as integer linear combinations and GCD are well-explained in detail within the document, ensuring that readers can follow along without confusion or ambiguity.
5. The assumptions made for solving this problem include: n is an integer; a and b have gcd(a,b)=1 (they're coprime); x and y must be non-negative integers if they exist as solutions to the equation \( ax + by = n \). These are reasonab

In [18]:

def synthesize_feedback(all_expert_responses):
    sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
    consolidated_feedback = {section: {"strengths": [], "weaknesses": [], "recommendations": [], "summaries": []} for section in sections}
    
    for expert in all_expert_responses:
        for section in sections:
            if section in expert["responses"]:
                response = expert["responses"][section]
                # Extracting strengths, weaknesses, recommendations, and summaries from each expert's response
                strengths = re.findall(r'\*\*Strengths\*\*:\n((?:\s+- .*\n)+)', response)
                weaknesses = re.findall(r'\*\*Weaknesses\*\*:\n((?:\s+- .*\n)+)', response)
                recommendations = re.findall(r'\*\*Recommendations\*\*:\n((?:\s+- .*\n)+)', response)
                summaries = re.findall(r'\*\*Summary\*\*:\n((?:\s+- .*\n)+)', response)

                if strengths:
                    consolidated_feedback[section]["strengths"].extend(strengths[0].strip().split('\n'))
                if weaknesses:
                    consolidated_feedback[section]["weaknesses"].extend(weaknesses[0].strip().split('\n'))
                if recommendations:
                    consolidated_feedback[section]["recommendations"].extend(recommendations[0].strip().split('\n'))
                if summaries:
                    consolidated_feedback[section]["summaries"].extend(summaries[0].strip().split('\n'))

    return consolidated_feedback

def analyze_reviewer(section_title, consolidated_feedback):
    context_window = 10000
    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback: {consolidated_feedback[section_title]}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="llama3-groq-tool-use", messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

#document_text = "..."  # Assign your document content here
#all_expert_responses = evaluate_all_experts(document_text)
consolidated_feedback = synthesize_feedback(all_expert_responses)
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, consolidated_feedback))

print("\n\n".join(final_review))

Section Introduction and Excellence:

1. **Strengths**:
   - The proposal demonstrates exceptional clarity in its objectives, methodology, and expected outcomes.
   - Strong collaboration among team members is evident throughout the document.

2. **Weaknesses**:
   - Some sections are overly ambitious or unclear about achievable goals within the project timeframe.
   - A more detailed budget breakdown could have been provided to ensure transparency.

3. **Recommendations**:
   - Refine objectives and outcomes for greater specificity, ensuring they align with the proposed methodology.
   - Provide a clearer timeline for expected milestones and deliverables in the budget section.

4. **Summary**:
The feedback highlights both strengths and areas of improvement regarding "Introduction and Excellence." While collaboration is strong, there's room to refine project goals and provide more detailed financial planning.

**Section Impact**

1. **Strengths:**
   - The project clearly defines its o